## 1. Configuración e Importaciones
En esta celda importamos las librerías y definimos las constantes del proyecto (nombres de datasets, algoritmos, etc.).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import pathlib as pl
from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, roc_auc_score, roc_curve, auc
from itertools import cycle

# CONFIGURACIÓN DE RUTAS
TYPES = ['original', 'estandarizado', 'normalizado']
VARIANTS = ['', '_PCA95', '_PCA80']
FOLDER_PREFIX = 'conj'
VALIDATING_FILE_REGEX = 'validating*.csv'
DATA_PATH = './kfolds_data'
MODEL_PATH = './trained_models'
MODEL_EXT = 'joblib'                       
PATH_PREDICCIONES = "./predicciones"
PATH_METRICAS = "./metricas"
PATH_GRAFICAS = "./graficas_roc"

CLASES = ['1-0', '0-1', '1/2-1/2']

os.makedirs(PATH_PREDICCIONES, exist_ok=True)
os.makedirs(PATH_METRICAS, exist_ok=True)
os.makedirs(PATH_GRAFICAS, exist_ok=True)

MODELOS = ["KNN", "SVM", "NaiveBayes", "RandomForest"]
ENSEMBLES = ["Ensemble_Votacion", "Ensemble_Media", "Ensemble_Mediana"]

## 2. Funciones Auxiliares (Carga y Guardado)
Necesitamos funciones para cargar los datos y modelos correspondientes a cada iteración y almacenar los resultados.

In [ ]:
def cargar_datos_test(filename):
    try:
        df = pd.read_csv(filename)
        X = df.drop('Result', axis=1)
        Y = df['Result']
        return X, Y
    except FileNotFoundError: return None, None

def calcular_metricas(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred) # Exactitud [cite: 64]
    f1 = f1_score(y_true, y_pred, average='macro') # F1-score [cite: 64]
    rec = recall_score(y_true, y_pred, average='macro') # Recall / Sensibilidad [cite: 64]
    prec = precision_score(y_true, y_pred, average='macro') # Precisión [cite: 64]
    
    cm = confusion_matrix(y_true, y_pred, labels=CLASES)
    FP = cm.sum(axis=0) - np.diag(cm)  
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    epsilon = 1e-7 
    spec_macro = np.mean(TN / (FP + TN + epsilon)) # Especificidad [cite: 64]
    fnr_macro = np.mean(FN / (TP + FN + epsilon))  # FNR [cite: 65]
    fpr_macro = np.mean(FP / (FP + TN + epsilon))  # FPR [cite: 65]
    
    auc_val = 0
    if y_proba is not None:
        try:
            auc_val = roc_auc_score(y_true, y_proba, multi_class='ovr', labels=CLASES)
        except: pass

    return {
        "Exactitud": acc, "F1": f1, "Sensibilidad": rec, "Recall": rec,
        "Precision": prec, "Especificidad": spec_macro,
        "FNR": fnr_macro, "FPR": fpr_macro, "AUC": auc_val
    }

def guardar_predicciones(y_true, y_pred, y_proba, path_file):
    df_pred = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})
    if y_proba is not None:
        for i, clase in enumerate(CLASES):
            df_pred[f'prob_{clase}'] = y_proba[:, i]
    df_pred.to_csv(path_file, index=False)

def plot_multiclass_roc(y_true, y_proba, dataset_name, method_name, fold):
    y_true_bin = label_binarize(y_true, classes=CLASES)
    n_classes = len(CLASES)
    fpr = dict(); tpr = dict(); roc_auc = dict()
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure()
    colors = cycle(['blue', 'red', 'green'])
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, label=f'ROC {CLASES[i]} (AUC = {roc_auc[i]:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'ROC {method_name} - {dataset_name} (Fold {fold})')
    plt.legend(loc="lower right")
    os.makedirs(f"{PATH_GRAFICAS}/{method_name}/{dataset_name}", exist_ok=True)
    plt.savefig(f"{PATH_GRAFICAS}/{method_name}/{dataset_name}/{method_name}{fold}_roc.png")
    plt.close()

## 3. Creación de predicciones y estadisticas de modelos base y ensembles
En esta sección iteramos sobre cada dataset y fold para generar las predicciones y métricas de los modelos base (KNN, SVM, NB, RF) utilizando los datos de test. Además, cuando los cuatro modelos se ejecutan correctamente, calculamos y evaluamos tres métodos de ensemble (Votación, Media y Mediana) combinando sus resultados para intentar mejorar el rendimiento final.

In [ ]:
for tipo in TYPES:
    for var in VARIANTS:
        dataset_name = f"{tipo}{var}"
        valid_route = pl.Path(f'{DATA_PATH}/{FOLDER_PREFIX}_{dataset_name}/')
        valid_csvs = sorted([x.name for x in valid_route.glob(VALIDATING_FILE_REGEX)])

        for fold_idx, csv_file in enumerate(valid_csvs):
            X_test, Y_test = cargar_datos_test(valid_route / csv_file)
            if X_test is None: continue
            
            fold_num = fold_idx + 1
            ensemble_probs = []
            ensemble_preds = []

            # 1. EVALUAR MODELOS BASE
            for model_name in MODELOS:
                m_path = f'{MODEL_PATH}/{model_name}/{dataset_name}/{model_name}{fold_num}_{dataset_name}.{MODEL_EXT}'
                modelo = joblib.load(m_path)
                
                y_pred = modelo.predict(X_test)
                y_proba = modelo.predict_proba(X_test)
                
                # Guardar métricas base
                metrics = calcular_metricas(Y_test, y_pred, y_proba)
                os.makedirs(f"{PATH_METRICAS}/{model_name}/{dataset_name}", exist_ok=True)
                os.makedirs(f"{PATH_PREDICCIONES}/{model_name}/{dataset_name}", exist_ok=True)
                
                pd.DataFrame([metrics]).to_csv(f"{PATH_METRICAS}/{model_name}/{dataset_name}/{model_name}{fold_num}_metrics.csv", index=False)
                guardar_predicciones(Y_test, y_pred, y_proba, f"{PATH_PREDICCIONES}/{model_name}/{dataset_name}/{model_name}{fold_num}_predicts.csv")
                plot_multiclass_roc(Y_test, y_proba, dataset_name, model_name, fold_num)
                
                ensemble_preds.append(y_pred)
                ensemble_probs.append(y_proba)

            # 2. CALCULAR ENSEMBLES
            # Obtenemos las clases directamente del último modelo para evitar errores de índice
            clases_modelo = modelo.classes_ 

            # A) Votación (Manejamos las modas de texto de forma segura)
            stack_preds = np.stack(ensemble_preds)
            y_pred_vot = pd.DataFrame(stack_preds).mode(axis=0).iloc[0].values
            
            # B) Media y C) Mediana de probabilidades
            stack_probs = np.stack(ensemble_probs)
            y_proba_avg = np.mean(stack_probs, axis=0)
            y_proba_med = np.median(stack_probs, axis=0)
            
            # Mapeo dinámico usando las clases del modelo
            y_pred_avg = clases_modelo[np.argmax(y_proba_avg, axis=1)]
            y_pred_med = clases_modelo[np.argmax(y_proba_med, axis=1)]

            # Guardar Ensembles
            ensembles_eval = [
                ("Ensemble_Votacion", y_pred_vot, y_proba_avg), 
                ("Ensemble_Media", y_pred_avg, y_proba_avg), 
                ("Ensemble_Mediana", y_pred_med, y_proba_med)
            ]

            for e_name, e_pred, e_prob in ensembles_eval:
                e_metrics = calcular_metricas(Y_test, e_pred, e_prob)
                os.makedirs(f"{PATH_METRICAS}/{e_name}/{dataset_name}", exist_ok=True)
                os.makedirs(f"{PATH_PREDICCIONES}/{e_name}/{dataset_name}", exist_ok=True)
                
                pd.DataFrame([e_metrics]).to_csv(f"{PATH_METRICAS}/{e_name}/{dataset_name}/{e_name}{fold_num}_metrics.csv", index=False)
                guardar_predicciones(Y_test, e_pred, e_prob, f"{PATH_PREDICCIONES}/{e_name}/{dataset_name}/{e_name}{fold_num}_predicts.csv")
                plot_multiclass_roc(Y_test, e_prob, dataset_name, e_name, fold_num)

print("Evaluación completada con éxito.")